# Full Pipeline Colab (Step 1-11)
Notebook ini menjalankan pipeline end-to-end dari audit data (Step 1) sampai draft laporan (Step 11).

## 0) Setup Runtime
Pilih GPU T4 di Colab: Runtime > Change runtime type > T4 GPU.

In [ ]:
import os
HF_TOKEN = 'hf_xxx_your_token_here'  # placeholder
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF_TOKEN placeholder set. Ganti dengan token kamu jika perlu.')

!git clone https://github.com/kzquandary/mbg-sentiment.git
%cd mbg-sentiment
!git checkout main
!python3 -m pip install -r requirements.txt
!nvidia-smi

## 1) Pilih Dataset Source
Default pakai `v2` (dataset terbaru relabel).

In [ ]:
# Toggle dataset source: v2 (default) | v1 | clean
DATASET_SOURCE = 'v2'
# DATASET_SOURCE = 'v1'
# DATASET_SOURCE = 'clean'
print('DATASET_SOURCE =', DATASET_SOURCE)

## 2) Run Full Pipeline Step 1-11
- Step 1: audit data
- Step 2: clean data
- Step 3-11: split, preprocess, EDA, baseline, model utama, tuning(optional), evaluasi, error analysis, laporan

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:64'
!python3 run_pipeline_full.py \
  --from-step 1 \
  --until-step 11 \
  --dataset-source "$DATASET_SOURCE" \
  --step7-config-json src/resources/step7_final_production.json \
  --step7-max-trials 1 \
  --run-class-multiplier

## 3) Lihat Hasil Utama Inline

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

metrics = json.load(open('outputs/final_metrics.json', encoding='utf-8'))
report = pd.read_csv('outputs/classification_report.csv')
cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)
runlog = json.load(open('outputs/pipeline_run_log.json', encoding='utf-8'))

print('Pipeline source:', runlog.get('dataset_source'))
print('Final Metrics')
display(pd.DataFrame([metrics])[['accuracy','precision_macro','recall_macro','f1_macro']])
print('Classification Report')
display(report)
print('Confusion Matrix (table)')
display(cm)
print('Confusion Matrix (figure)')
display(Image(filename='outputs/figures/confusion_matrix.png'))

## 4) Artifact Penting
- `outputs/pipeline_run_log.json`
- `outputs/final_metrics.json`
- `outputs/classification_report.csv`
- `outputs/confusion_matrix.csv`
- `outputs/error_analysis_summary.md`
- `outputs/bab4_hasil_otomatis.md`
- `outputs/bab5_kesimpulan_saran_otomatis.md`